## Quickstart
Concise examples for code construction, syndrome-extraction IR, stim generation, and scheduling.

### 1) CSS Codes

In [1]:
from sim.qecc.css import CSS_Code

# Instantiate from parity-check matrices
Hx = [
    [1, 1, 0, 0],
    [0, 1, 1, 0],
]
Hz = [
    [0, 1, 1, 1],
    [1, 0, 0, 1],
]
css_from_h = CSS_Code(Hx=Hx, Hz=Hz)

# Instantiate from explicit X/Z check supports
x_checks = [[0, 1], [1, 2]]
z_checks = [[1, 2, 3], [0, 3]]
css_from_checks = CSS_Code(x_checks=x_checks, z_checks=z_checks, num_data_q=4)

print("[CSS from H] n, rx, rz =", css_from_h.num_data_q, css_from_h.num_x_check, css_from_h.num_z_check)
print("[CSS from checks] n, rx, rz =", css_from_checks.num_data_q, css_from_checks.num_x_check, css_from_checks.num_z_check)

[CSS from H] n, rx, rz = 4 2 2
[CSS from checks] n, rx, rz = 4 2 2


### 2) BB Codes

In [ ]:
from sim.qecc.qalp import BB_Code

# BB code over Z_l x Z_m with two polynomials A, B
bb_code = BB_Code(
    l=3,
    m=3,
    A_poly=[[0, 0], [1, 0]],
    B_poly=[[0, 0], [0, 1]],
)
print("BB code n, rx, rz =", bb_code.num_data_q, bb_code.num_x_check, bb_code.num_z_check)

BB code n, rx, rz = 18 9 9


### 3) TT Codes

In [ ]:
from sim.qecc.qalp import TT_Code

# TT code with three polynomials A, B, C
tt_code = TT_Code( # First row in Table III of Menon et al. Phys. Rev. X 16, 021014
    l=2,
    m=2,
    n=3,
    A_poly=[[0, 1, 1], [1, 0, 0]],
    B_poly=[[0, 0, 2], [0, 1, 0]],
    C_poly=[[1, 1, 1], [0, 0, 1]],
) # This is a [[48, 6, 4]] code
print("TT code n, rx, rz =", tt_code.num_data_q, tt_code.num_x_check, tt_code.num_z_check)

TT code n, rx, rz = 36 12 36


### 4) Stim Generation + Scheduling IR
The SE schedule IR is `(s_x_check, s_z_check, depth)` and is the bridge between scheduling (`sim.scheduling`) and circuit synthesis (`sim.qecc.stim`).

In [ ]:
import stim
from sim.qecc.stim import generate_stim_str
from sim.scheduling import random_schedule

# Sample one valid schedule and load it into the code's IR
s_x, s_z = random_schedule(tt_code)
tt_code.set_se_schedule_ir(s_x, s_z)

p_2q_projection = 0.01 * 4 / 15
p_1q_projection = 0.001 * 2 / 3

stim_str = generate_stim_str(
    tt_code, rounds=12, obs_type='Z', 
    p_2q_channel=f'PAULI_CHANNEL_2({p_2q_projection},0,0,{p_2q_projection},{p_2q_projection},0,0,0,0,0,0,0,0,0,0)', 
    p_idle_channel=None,
    p_meas=0.001
)
stim.Circuit(stim_str).detector_error_model(approximate_disjoint_errors=True)
print(stim_str.splitlines()[:8])

dem = stim.Circuit(stim_str).detector_error_model(approximate_disjoint_errors=True)
print("Detector error model terms:", len(str(dem).splitlines()))

['R 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83', 'X_ERROR(0.001) 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83', 'RX 36 37 38 39 40 41 42 43 44 45 46 47', 'Z_ERROR(0.001) 36 37 38 39 40 41 42 43 44 45 46 47', 'TICK', '', 'CX 36 14 37 12 38 13 39 17 40 15 41 16 42 20 43 18 44 19 45 23 46 21 47 22 3 48 4 49 5 50 0 51 1 52 2 53 9 54 10 55 11 56 6 57 7 58 8 59 27 72 28 73 29 74 24 75 25 76 26 77 33 78 34 79 35 80 30 81 31 82 32 83', 'PAULI_CHANNEL_2(0.0026666666666666666,0,0,0.0026666666666666666,0.0026666666666666666,0,0,0,0,0,0,0,0,0,0) 36 14 37 12 38 13 39 17 40 15 41 16 42 20 43 18 44 19 45 23 46 21 47 22 3 48 4 49 5 50 0 51 1 52 2 53 9 54 10 55 11 56 6 57 7 58 8 59 27

### 6) `sim.scheduling`: all schedules + random schedule

In [ ]:
from sim.scheduling import all_schdules, random_schedule

# Enumerate all schedules for the [[56,6,6]] instance
all_scheds = all_schdules(tt_code)
print("Number of schedules:", len(all_scheds))

# Draw one random schedule from the same space
s_x_rand, s_z_rand = random_schedule(tt_code)
tt_code.set_se_schedule_ir(s_x_rand, s_z_rand)
print("Random schedule depth:", tt_code.depth)

Number of schedules: 48
Random schedule depth: 6
